# Layer 8 — Digital Twin Simulation Engine
**Bengaluru Traffic Intelligence Platform (BTIP)**

This notebook walks through the Layer 8 simulation pipeline built in
`backend/simulation/deterrence_model.py`, `graph_diffusion.py`, and
`digital_twin.py`, and exercised end-to-end via `POST /api/v1/simulation`
(`backend/api/routes/simulation.py`). It covers:

1. Deterrence decay curve — officers → violation reduction, and time decay after officers leave
2. Graph diffusion — congestion relief propagating to 1-hop / 2-hop neighbors
3. Digital twin orchestration — full before/after what-if scenario
4. Confidence band (P10/P50/P90) via Monte Carlo on the deterrence constant
5. Sanity checks: monotonicity, latency, spillover vs direct effect

**Prerequisite:** Layer 6 (`congestion_scorer.py`) should have produced
`data/processed/junction_congestion_scores.parquet`, and the OSM graph at
`data/external/bengaluru_osm_graph.graphml` should be present, for the
*real-data* sections below. If either is missing, this notebook falls
back to a small synthetic graph/baseline (clearly marked) so it still
runs end-to-end.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import networkx as nx

PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / "backend").exists() is False else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from backend.simulation.deterrence_model import DeterrenceModel, DeterrenceParams
from backend.simulation.graph_diffusion import GraphDiffusion
from backend.simulation.digital_twin import DigitalTwin

plt.rcParams["figure.facecolor"] = "#0B0E14"
plt.rcParams["axes.facecolor"] = "#141820"
plt.rcParams["axes.edgecolor"] = "#666B7A"
plt.rcParams["axes.labelcolor"] = "#E8E8E8"
plt.rcParams["text.color"] = "#E8E8E8"
plt.rcParams["xtick.color"] = "#E8E8E8"
plt.rcParams["ytick.color"] = "#E8E8E8"
plt.rcParams["figure.figsize"] = (7, 5)

CYAN, AMBER, RED, GREEN = "#00D4FF", "#FFB020", "#FF4444", "#00A86B"

## 1. Deterrence curve — officers → violation reduction

`DeterrenceModel.fit_k()` tries to fit `k` from historical `officer_count`
variation in `feature_store.parquet`. If that column isn't present yet
(expected at this stage of the build), it falls back to the documented
default `k = 0.3` and logs why — this is graceful, not an error.

In [ ]:
dm = DeterrenceModel()
params = dm.fit_k()  # attempts historical fit, falls back to default if unavailable

print(f"k = {params.k:.3f}")
print(f"base_rate = {params.base_rate:.3f}")
print(f"fitted_from_data = {params.fitted_from_data}")
if params.fitted_from_data:
    print(f"n_samples_used = {params.n_samples_used}")
else:
    print("Using default k=0.3 (no 'officer_count' history in feature_store yet).")

In [ ]:
officer_counts = np.arange(0, 8)
reductions = [dm.reduction_pct(n) * 100 for n in officer_counts]

fig, ax = plt.subplots()
ax.plot(officer_counts, reductions, marker="o", color=CYAN, linewidth=2)
ax.set_xlabel("Officers deployed")
ax.set_ylabel("Violation reduction (%)")
ax.set_title(f"Deterrence Decay Curve  (k={dm.params.k:.3f})")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print("Marginal gain per additional officer (diminishing returns):")
for i in range(1, len(reductions)):
    print(f"  {i-1} → {i} officers: +{reductions[i] - reductions[i-1]:.1f}pp")

### Time decay

Once officers leave a zone, the deterrence effect decays exponentially
with a 2-hour half-life — by ~6-7h the effect is negligible.

In [ ]:
hours = np.linspace(0, 8, 50)
base_effect = dm.reduction_pct(3)
decayed = [dm.time_decayed_effect(base_effect, h) * 100 for h in hours]

fig, ax = plt.subplots()
ax.plot(hours, decayed, color=AMBER, linewidth=2)
ax.axhline(base_effect * 100, color=CYAN, linestyle=":", linewidth=1, label="Effect while present")
ax.axvline(dm.params.half_life_hours, color=RED, linestyle="--", linewidth=1,
           label=f"Half-life = {dm.params.half_life_hours}h")
ax.set_xlabel("Hours since officers left")
ax.set_ylabel("Residual violation reduction (%)")
ax.set_title("Deterrence Time Decay  (3 officers, post-departure)")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Graph diffusion — congestion relief propagation

Tries to load the real Bengaluru OSM graph (Layer 6). Falls back to a
small synthetic road graph (clearly labelled) if the graphml file or
`networkx` graph isn't available yet, so this section always runs.

In [ ]:
GRAPH_PATH = PROJECT_ROOT / "data" / "external" / "bengaluru_osm_graph.graphml"

using_real_graph = False
try:
    from backend.decision.graph_intelligence import GraphIntelligence
    gi = GraphIntelligence()
    gi.load_graph()
    gd = GraphDiffusion(graph_intelligence=gi)
    using_real_graph = True
    print(f"Loaded real OSM graph: {gi.G.number_of_nodes():,} nodes, {gi.G.number_of_edges():,} edges")
except (FileNotFoundError, Exception) as e:
    print(f"Real OSM graph unavailable ({type(e).__name__}: {e})")
    print("Falling back to a synthetic 6-node demo graph for this notebook.")

    G_demo = nx.Graph()
    G_demo.add_edge("SilkBoard", "HSRLayout", length=400.0)
    G_demo.add_edge("SilkBoard", "Electronic City", length=800.0)
    G_demo.add_edge("HSRLayout", "Koramangala", length=1200.0)
    G_demo.add_edge("Electronic City", "Bommanahalli", length=1500.0)

    class _DemoGI:
        def __init__(self, G):
            self.G = G
        def edge_weight(self, u, v):
            if not self.G.has_edge(u, v):
                return 0.0
            length = self.G[u][v].get("length", 100.0)
            return max(0.0, min(1.0, 1.0 - length / 2000.0))

    gd = GraphDiffusion(graph_intelligence=_DemoGI(G_demo))

In [ ]:
patrolled_zone = "SilkBoard" if not using_real_graph else next(iter(gi.G.nodes()))
spillover = gd.propagate(patrolled_zone, direct_reduction=0.6, max_hops=2)

print(f"Patrolling: {patrolled_zone}  (direct_reduction=60%)\n")
print("Spillover relief to neighbors:")
for zone, relief in sorted(spillover.items(), key=lambda x: -x[1]):
    print(f"  {zone}: {relief*100:.2f}% relief")

In [ ]:
if spillover:
    fig, ax = plt.subplots()
    zones = list(spillover.keys())
    reliefs = [spillover[z] * 100 for z in zones]
    colors = [CYAN if r > 10 else AMBER for r in reliefs]
    ax.barh(zones, reliefs, color=colors)
    ax.set_xlabel("Spillover relief (%)")
    ax.set_title(f"Graph Diffusion — Spillover from {patrolled_zone}")
    plt.tight_layout()
    plt.show()
else:
    print("No spillover neighbors found for this node (may be isolated in the graph).")

## 3. Digital twin — full what-if scenario

Combines deterrence + graph diffusion + baseline state from Layer 6's
`junction_congestion_scores.parquet`. Falls back to a small synthetic
baseline if that file hasn't been generated yet (e.g. `congestion_scorer.py`
still running in the background).

In [ ]:
twin = DigitalTwin(deterrence_model=dm, graph_diffusion=gd)

CONGESTION_PATH = PROJECT_ROOT / "data" / "processed" / "junction_congestion_scores.parquet"
using_real_baseline = CONGESTION_PATH.exists()

if not using_real_baseline:
    print(f"{CONGESTION_PATH} not found yet — using synthetic baseline for this walkthrough.")
    print("(Re-run this notebook once Layer 6's congestion_scorer.py finishes.)\n")
    synthetic_baseline = {
        "SilkBoard": {"violation_rate": 42.0, "congestion_score": 88.0},
        "HSRLayout": {"violation_rate": 27.0, "congestion_score": 61.0},
        "Electronic City": {"violation_rate": 19.0, "congestion_score": 54.0},
        "Koramangala": {"violation_rate": 22.0, "congestion_score": 49.0},
        "Bommanahalli": {"violation_rate": 14.0, "congestion_score": 38.0},
    }
    twin._load_baseline_state = lambda: synthetic_baseline
    scenario_zone = "SilkBoard"
else:
    print(f"Loaded real congestion baseline from {CONGESTION_PATH}")
    cong_df = pl.read_parquet(str(CONGESTION_PATH))
    scenario_zone = str(cong_df.sort("congestion_score", descending=True)["junction_id"][0])
    print(f"Top congestion zone: {scenario_zone}")

In [ ]:
result = twin.run_scenario(
    zone_allocations={scenario_zone: 3},
    shift="Evening",
    date="2024-01-15",
)

print(f"Scenario: 3 officers at {scenario_zone}, Evening shift\n")
print(f"Total violations before: {result['total_violations_before']}")
print(f"Total violations after:  {result['total_violations_after']}")
print(f"Reduction:               {result['reduction_pct']}%")
print(f"Congestion improvement:  {result['congestion_improvement_pct']}%")
print(f"Affected junctions:      {result['affected_junction_count']}")
print(f"Latency:                 {result['latency_seconds']}s  (target < 3s)")
print(f"Confidence band:         {result['confidence_band']}")

In [ ]:
pj = pl.DataFrame(result["per_junction"])
pj

### Before / after comparison

In [ ]:
junctions = [j["junction_id"] for j in result["per_junction"]]
before = [j["violation_rate_before"] for j in result["per_junction"]]
after = [j["violation_rate_after"] for j in result["per_junction"]]

x = np.arange(len(junctions))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, before, width, label="Before", color="#666B7A")
ax.bar(x + width/2, after, width, label="After", color=CYAN)
ax.set_xticks(x)
ax.set_xticklabels(junctions, rotation=30, ha="right")
ax.set_ylabel("Violation rate (proxy)")
ax.set_title(f"Before / After — 3 officers at {scenario_zone}")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Confidence band — Monte Carlo on deterrence k

100 runs with ±10% noise on `k`. Per the Layer 8 verification checklist,
P10 < P50 < P90 must always hold.

In [ ]:
band = result["confidence_band"]
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.barh([0], [band["p90"] - band["p10"]], left=band["p10"], color=CYAN, alpha=0.3, height=0.4)
ax.axvline(band["p50"], color=CYAN, linewidth=2, label="P50 (median)")
ax.axvline(band["p10"], color="#666B7A", linestyle="--", linewidth=1, label="P10")
ax.axvline(band["p90"], color="#666B7A", linestyle="--", linewidth=1, label="P90")
ax.set_yticks([])
ax.set_xlabel("Total reduction (%)")
ax.set_title("Confidence Band — 100 Monte Carlo runs (±10% noise on k)")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.tight_layout()
plt.show()

assert band["p10"] <= band["p50"] <= band["p90"], "P10 < P50 < P90 must hold"
print("OK — P10 ≤ P50 ≤ P90 holds.")

## 5. Sanity checks

- **Monotonicity:** adding more officers must never decrease reduction.
- **Latency:** must stay under 3s for the real-time Simulation Lab demo.
- **Spillover vs direct effect:** directly patrolled zones should see a
  larger reduction than zones only receiving spillover relief.

In [ ]:
officer_sweep = [1, 2, 3, 4, 5]
reductions_sweep = []
latencies = []

for n in officer_sweep:
    r = twin.run_scenario(
        zone_allocations={scenario_zone: n}, shift="Evening", date="2024-01-15",
        run_confidence_band=False,
    )
    reductions_sweep.append(r["reduction_pct"])
    latencies.append(r["latency_seconds"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(officer_sweep, reductions_sweep, marker="o", color=GREEN)
ax1.set_xlabel("Officers at zone")
ax1.set_ylabel("Total reduction (%)")
ax1.set_title("Monotonicity check")
ax1.grid(alpha=0.2)

ax2.bar([str(n) for n in officer_sweep], latencies, color=AMBER)
ax2.axhline(3.0, color=RED, linestyle="--", label="3s target")
ax2.set_xlabel("Officers at zone")
ax2.set_ylabel("Latency (s)")
ax2.set_title("Latency check")
ax2.legend()
plt.tight_layout()
plt.show()

assert reductions_sweep == sorted(reductions_sweep), "reduction must be monotonically non-decreasing in officer count"
assert all(l < 3.0 for l in latencies), "all runs must complete under 3s"
print("OK — monotonicity and latency checks pass.")

In [ ]:
direct = [j for j in result["per_junction"] if j["is_directly_patrolled"]]
spillover_only = [j for j in result["per_junction"] if not j["is_directly_patrolled"]]

if direct and spillover_only:
    max_direct = max(j["reduction_pct"] for j in direct)
    max_spillover = max(j["reduction_pct"] for j in spillover_only)
    print(f"Max direct-patrol reduction:   {max_direct}%")
    print(f"Max spillover-only reduction:  {max_spillover}%")
    assert max_direct > max_spillover, "direct patrol effect should exceed spillover-only effect"
    print("OK — direct effect exceeds spillover effect, as expected.")
else:
    print("Not enough affected junctions in this scenario to compare direct vs spillover.")

## Summary

- **Deterrence curve** — concave, diminishing-returns shape; `k` is fit
  from historical officer-count variation when available, else falls back
  to the documented default (`k=0.3`).
- **Graph diffusion** — relief reaches 1-hop and 2-hop neighbors and
  decays correctly with distance/hops; directly patrolled zones are
  excluded from their own spillover.
- **Digital twin** — produces a complete before/after state per junction,
  with `reduction_pct`, `congestion_improvement_pct`, and a Monte Carlo
  P10/P50/P90 confidence band that always satisfies P10 ≤ P50 ≤ P90.
- **Sanity checks** — monotonicity (more officers never hurts), latency
  (< 3s, real-time-demo safe), and direct-vs-spillover ordering all pass.
- This notebook runs end-to-end even before Layer 6's
  `junction_congestion_scores.parquet` / OSM graph are finished — swap to
  the real-data branches automatically once those files exist on disk.